In [11]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import hstack

# 1. Load your Test Dataset
test_df = pd.read_csv('arvyax_test_inputs_120.xlsx - Sheet1.csv') 

# 2. Load the trained AI (Artifacts)
with open("model_artifacts.pkl", "rb") as f:
    arts = pickle.load(f)
    state_model = arts["state_model"]
    intensity_model = arts["intensity_model"]
    meta_preprocessor = arts["metadata_preprocessor"]
    tfidf = arts["tfidf"]
    le = arts["le"]
    # These match your CSV columns
    numerical_cols = ["duration_min", "sleep_hours", "energy_level", "stress_level"]
    categorical_cols = ["ambience_type", "time_of_day", "previous_day_mood", 
                        "face_emotion_hint", "reflection_quality"]

print(f"--- Processing {len(test_df)} test samples ---")

# 3. Preprocessing
# Clean Text
test_df["journal_text_clean"] = test_df["journal_text"].fillna("missing").str.lower()
X_text = tfidf.transform(test_df["journal_text_clean"])

# Transform Metadata
X_meta = meta_preprocessor.transform(test_df[numerical_cols + categorical_cols])

# Combine everything
X_final = hstack([X_text, X_meta]).toarray()

# 4. Predict
state_preds_encoded = state_model.predict(X_final)
test_df["predicted_state"] = le.inverse_transform(state_preds_encoded)

intensity_raw = intensity_model.predict(X_final)
test_df["predicted_intensity"] = np.clip(np.round(intensity_raw), 1, 5).astype(int)

# 5. Save Results
output_file = "test_results_with_predictions.csv"
test_df.to_csv(output_file, index=False)

print(f"Done! Open '{output_file}' to see the AI's guesses.")

# 6. Show a small sample of the results
print("\nSample Predictions:")
print(test_df[["id", "journal_text", "predicted_state", "predicted_intensity"]].head(5))

--- Processing 120 test samples ---
Done! Open 'test_results_with_predictions.csv' to see the AI's guesses.

Sample Predictions:
      id                                       journal_text predicted_state  \
0  10001  woke up feeling more organized mentally. i was...         focused   
1  10002  started off distracted most of the time. this ...           mixed   
2  10003                                     kinda calm ...        restless   
3  10004  after the session i felt able to think straigh...         focused   
4  10005  lowkey felt pretty grounded. i had to restart ...            calm   

   predicted_intensity  
0                    3  
1                    4  
2                    4  
3                    3  
4                    3  
